# ING Feature Exploration

Interactive exploration of features collected by the Hyperliquid ingestor.

**Contents:**
1. Data Loading & Summary
2. Feature Time Series
3. Distribution Analysis
4. Correlation Analysis
5. Event Detection
6. PCA / Dimensionality Reduction

In [ ]:
# Setup
import sys
sys.path.insert(0, '../scripts')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import visualization module
from viz import (
    load_data, load_recent, summarize, get_symbols,
    plot_features, plot_feature_panel, plot_rolling_stats,
    plot_events, detect_events, plot_event_study,
    plot_correlation_matrix, plot_scatter_matrix, plot_pca, find_correlated_pairs,
    plot_distributions, plot_qq, plot_outliers, distribution_summary
)

%matplotlib inline
plt.style.use('dark_background')

print("Visualization module loaded!")

## 1. Data Loading

In [ ]:
# Load all data
DATA_DIR = '../data/features'

df = load_data(DATA_DIR)
summarize(df)

In [ ]:
# Get available symbols
symbols = get_symbols(df)
print(f"Available symbols: {symbols}")

# Select symbol for analysis
SYMBOL = symbols[0] if symbols else None
print(f"\nAnalyzing: {SYMBOL}")

In [ ]:
# Preview data
df.head()

In [ ]:
# List all columns
print("Columns:")
for i, col in enumerate(df.columns):
    print(f"  {i:3d}. {col}")

## 2. Feature Time Series

In [ ]:
# Plot key features together (normalized for comparison)
key_features = ['vpin_10', 'qty_l1', 'returns_1m', 'kyle_lambda_100']
# Filter to existing columns
key_features = [f for f in key_features if f in df.columns]

if key_features:
    fig = plot_features(df, key_features, symbol=SYMBOL, normalize=True)
    plt.show()
else:
    print("Key features not found. Available columns:")
    print([c for c in df.columns if c not in ['timestamp_ns', 'timestamp', 'symbol', 'sequence_id', 'datetime']][:10])

In [ ]:
# Feature panel - each feature in separate subplot
panel_features = df.columns[3:8].tolist()  # First 5 feature columns

fig = plot_feature_panel(df, panel_features, symbol=SYMBOL)
plt.show()

In [ ]:
# Rolling statistics for a single feature
feature_to_analyze = df.columns[3]  # First feature column
print(f"Analyzing: {feature_to_analyze}")

fig = plot_rolling_stats(df, feature_to_analyze, windows=[10, 50, 200], symbol=SYMBOL)
plt.show()

## 3. Distribution Analysis

In [ ]:
# Distribution summary table
feature_cols = [c for c in df.columns if c not in ['timestamp_ns', 'timestamp', 'symbol', 'sequence_id', 'datetime']]
summary = distribution_summary(df, feature_cols[:20])  # First 20 features
summary.round(4)

In [ ]:
# Histograms
features_to_plot = feature_cols[:6]  # First 6 features

fig = plot_distributions(df, features_to_plot, symbol=SYMBOL)
plt.show()

In [ ]:
# Q-Q plots (check normality)
fig = plot_qq(df, features_to_plot[:4], symbol=SYMBOL)
plt.show()

In [ ]:
# Outlier analysis
fig = plot_outliers(df, features_to_plot, symbol=SYMBOL, threshold=3.0)
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation matrix
fig = plot_correlation_matrix(df, features=feature_cols[:15], symbol=SYMBOL, annot=False)
plt.show()

In [ ]:
# Find highly correlated pairs
corr_pairs = find_correlated_pairs(df, threshold=0.7, features=feature_cols)
print("Highly correlated feature pairs (|r| > 0.7):")
corr_pairs.head(20)

In [ ]:
# Scatter matrix for selected features
scatter_features = feature_cols[:4]  # First 4 features

fig = plot_scatter_matrix(df, scatter_features, symbol=SYMBOL, sample_size=3000)
plt.show()

## 5. Event Detection

In [ ]:
# Detect and plot events
event_features = [f for f in ['vpin_10', 'qty_l1', 'returns_1m'] if f in df.columns]

if event_features:
    fig = plot_events(df, event_features=event_features, symbol=SYMBOL, threshold_std=2.5)
    plt.show()
else:
    # Use first few feature columns
    event_features = feature_cols[:3]
    fig = plot_events(df, event_features=event_features, symbol=SYMBOL, threshold_std=2.5)
    plt.show()

In [ ]:
# Detect events in a specific feature
feature_for_events = feature_cols[0]
events = detect_events(df[df['symbol'] == SYMBOL] if SYMBOL else df, 
                       feature_for_events, threshold_std=2.5)

print(f"Detected {len(events)} events in {feature_for_events}")
for e in events[:5]:
    print(f"  {e.timestamp}: z={e.zscore:.2f}, value={e.value:.4g}")

In [ ]:
# Event study - average behavior around events
if events:
    try:
        study_feature = feature_cols[1] if len(feature_cols) > 1 else feature_cols[0]
        fig = plot_event_study(
            df[df['symbol'] == SYMBOL] if SYMBOL else df,
            events,
            feature=study_feature,
            window_before=30,
            window_after=60
        )
        plt.show()
    except Exception as e:
        print(f"Could not generate event study: {e}")

## 6. PCA / Dimensionality Reduction

In [ ]:
# PCA projection
try:
    fig = plot_pca(df, features=feature_cols[:20], n_components=2, symbol=SYMBOL)
    plt.show()
except Exception as e:
    print(f"PCA failed: {e}")
    print("This may be due to NaN values or insufficient data.")

In [ ]:
# PCA with explained variance
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Prepare data
X = df[feature_cols].dropna()
if len(X) > 100:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    pca = PCA(n_components=min(20, len(feature_cols)))
    pca.fit(X_scaled)
    
    # Plot explained variance
    fig, ax = plt.subplots(figsize=(10, 5))
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    ax.bar(range(1, len(cumvar)+1), pca.explained_variance_ratio_, alpha=0.7, label='Individual')
    ax.plot(range(1, len(cumvar)+1), cumvar, 'r-o', label='Cumulative')
    ax.axhline(0.9, color='green', linestyle='--', alpha=0.5, label='90% threshold')
    ax.set_xlabel('Principal Component')
    ax.set_ylabel('Explained Variance Ratio')
    ax.set_title('PCA Explained Variance')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    
    # How many components for 90%?
    n_90 = np.argmax(cumvar >= 0.9) + 1
    print(f"Components needed for 90% variance: {n_90}")
else:
    print("Insufficient non-NaN data for PCA")

## 7. Custom Analysis

Add your own analysis below:

In [ ]:
# Your custom analysis here
# Example: Compare feature distributions across symbols

if len(symbols) > 1:
    feature = feature_cols[0]
    fig, ax = plt.subplots(figsize=(10, 5))
    
    for sym in symbols:
        data = df[df['symbol'] == sym][feature].dropna()
        ax.hist(data, bins=50, alpha=0.5, label=sym)
    
    ax.set_xlabel(feature)
    ax.set_ylabel('Count')
    ax.set_title(f'{feature} Distribution by Symbol')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()